# 18 · FPT levels and relative scope gaps — Gate 19

**Purpose.** Address the scale-comparability diagnostic by reporting the female and male mean first-passage-time (FPT) levels behind every absolute gap and by normalizing that gap by the two-group mean:

$$
G_{\mathrm{rel}}=\frac{M^{(w)}-M^{(m)}}{\{M^{(w)}+M^{(m)}\}/2}.
$$

This notebook performs no model re-estimation. It reads the validated point estimates from Gate 12, verifies their arithmetic identities, and produces descriptive normalized contrasts for Bogotá D.C. internal and Bogotá–Region. Statistical inference remains attached to the preregistered absolute gap; relative gaps in this gate are descriptive diagnostics.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil

import numpy as np
import pandas as pd

ROOT = Path("/content/colombia-stochastic-access")
OUT = ROOT / "outputs/bogota_em2023/gate19"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
OUT.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Environment outside Colab or Drive not mounted: {exc}")

G12 = Path(os.environ.get("GATE12_DIR_OVERRIDE", DRIVE_BASE / "results/gate12"))
paths = {
    "gate12": G12 / "gate_12.json",
    "scope_points": G12 / "scope_comparison_point_estimates.csv",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing validated inputs: " + ", ".join(missing))

gate12 = json.loads(paths["gate12"].read_text(encoding="utf-8"))
if not gate12.get("gate", {}).get("gate_12_passed", False):
    raise RuntimeError("Gate 12 is not approved")

print("Validated dependency: Gate 12")
print("Output:", OUT)


Mounted at /content/drive
Validated dependency: Gate 12
Output: /content/colombia-stochastic-access/outputs/bogota_em2023/gate19


## 1. Recover and audit the group-specific FPT levels

The stored female-minus-male gap must reproduce the subtraction of the two reported levels at machine precision.


In [2]:
points = pd.read_csv(paths["scope_points"])
required = {
    "scope", "purpose_group", "male_total_fpt", "female_total_fpt",
    "female_minus_male_total",
}
absent = sorted(required - set(points.columns))
if absent:
    raise KeyError(f"Missing columns in scope point estimates: {absent}")

levels = points[list(required)].copy()
levels["recomputed_absolute_gap"] = levels["female_total_fpt"] - levels["male_total_fpt"]
levels["two_group_mean_fpt"] = (
    levels["female_total_fpt"] + levels["male_total_fpt"]
) / 2.0
levels["relative_gap"] = (
    levels["female_minus_male_total"] / levels["two_group_mean_fpt"]
)
levels["relative_gap_percent"] = 100.0 * levels["relative_gap"]
levels["gap_identity_error"] = (
    levels["female_minus_male_total"] - levels["recomputed_absolute_gap"]
).abs()

scope_order = ["bogota_dc_internal", "bogota_region"]
purpose_order = ["employment", "education", "health"]
levels["scope"] = pd.Categorical(levels["scope"], scope_order, ordered=True)
levels["purpose_group"] = pd.Categorical(
    levels["purpose_group"], purpose_order, ordered=True
)
levels = levels.sort_values(["purpose_group", "scope"]).reset_index(drop=True)

display(levels[[
    "scope", "purpose_group", "male_total_fpt", "female_total_fpt",
    "female_minus_male_total", "two_group_mean_fpt", "relative_gap_percent",
]].round(3))
print("Maximum gap identity error:", levels["gap_identity_error"].max())


,scope,purpose_group,male_total_fpt,female_total_fpt,female_minus_male_total,two_group_mean_fpt,relative_gap_percent
0,bogota_dc_internal,employment,3.616,4.095,0.479,3.855,12.426
1,bogota_region,employment,4.111,5.110,0.999,4.611,21.662
2,bogota_dc_internal,education,3.789,4.187,0.399,3.988,9.995
3,bogota_region,education,4.306,5.133,0.827,4.719,17.532
4,bogota_dc_internal,health,5.045,5.285,0.240,5.165,4.655
5,bogota_region,health,6.257,6.963,0.705,6.610,10.671


Maximum gap identity error: 1.1102230246251565e-16


## 2. Descriptive amplification on the relative scale

A positive regional-minus-district difference indicates that territorial amplification remains after normalizing the absolute gap by the corresponding mean FPT level.


In [3]:
wide = levels.pivot(index="purpose_group", columns="scope")
contrast_rows = []
for purpose in purpose_order:
    city = float(wide.loc[purpose, ("relative_gap", "bogota_dc_internal")])
    region = float(wide.loc[purpose, ("relative_gap", "bogota_region")])
    contrast_rows.append({
        "purpose_group": purpose,
        "district_relative_gap": city,
        "regional_relative_gap": region,
        "regional_minus_district_relative_gap": region - city,
        "regional_to_district_relative_gap_ratio": region / city,
    })
relative_contrasts = pd.DataFrame(contrast_rows)
display(relative_contrasts.round(4))

export_columns = [
    "scope", "purpose_group", "male_total_fpt", "female_total_fpt",
    "female_minus_male_total", "two_group_mean_fpt", "relative_gap",
    "relative_gap_percent", "gap_identity_error",
]
levels[export_columns].to_csv(OUT / "fpt_levels_relative_gaps.csv", index=False)
relative_contrasts.to_csv(OUT / "relative_gap_scope_contrasts.csv", index=False)

scope_labels = {
    "bogota_dc_internal": "Bogotá D.C. internal",
    "bogota_region": "Bogotá–Region",
}
purpose_labels = {
    "employment": "Employment", "education": "Education", "health": "Health",
}
table = levels.copy()
table["Scope"] = table["scope"].astype("string").map(scope_labels)
table["Purpose"] = table["purpose_group"].astype("string").map(purpose_labels)
table = table.rename(columns={
    "male_total_fpt": "Male mean FPT",
    "female_total_fpt": "Female mean FPT",
    "female_minus_male_total": "Absolute gap",
    "relative_gap_percent": "Relative gap (percent)",
})[["Scope", "Purpose", "Male mean FPT", "Female mean FPT", "Absolute gap", "Relative gap (percent)"]]
table.to_csv(OUT / "table_fpt_levels_relative_gaps.csv", index=False)
table.to_latex(
    OUT / "table_fpt_levels_relative_gaps.tex",
    index=False,
    float_format="%.3f",
    caption=("Group-specific mean FPT levels and descriptive female-minus-male gaps "
             "on absolute and normalized scales."),
    label="tab:fpt_levels_relative_gaps",
)
display(table.round(3))


,purpose_group,district_relative_gap,regional_relative_gap,regional_minus_district_relative_gap,regional_to_district_relative_gap_ratio
0,employment,0.1243,0.2166,0.0924,1.7432
1,education,0.1000,0.1753,0.0754,1.7540
2,health,0.0466,0.1067,0.0602,2.2922


,Scope,Purpose,Male mean FPT,Female mean FPT,Absolute gap,Relative gap (percent)
0,Bogotá D.C. internal,Employment,3.616,4.095,0.479,12.426
1,Bogotá–Region,Employment,4.111,5.110,0.999,21.662
2,Bogotá D.C. internal,Education,3.789,4.187,0.399,9.995
3,Bogotá–Region,Education,4.306,5.133,0.827,17.532
4,Bogotá D.C. internal,Health,5.045,5.285,0.240,4.655
5,Bogotá–Region,Health,6.257,6.963,0.705,10.671


## 3. Gate checks and persistent artifacts


In [4]:
identity_tolerance = 1e-10
checks = {
    "gate12_dependency_valid": True,
    "six_scope_purpose_rows_present": bool(len(levels) == 6),
    "all_required_scope_purpose_pairs_unique": bool(
        levels.groupby(["scope", "purpose_group"], observed=True).size().eq(1).all()
    ),
    "absolute_gap_identity_valid": bool(
        levels["gap_identity_error"].max() <= identity_tolerance
    ),
    "all_levels_finite_and_positive": bool(
        np.isfinite(levels[["male_total_fpt", "female_total_fpt"]]).all().all()
        and levels[["male_total_fpt", "female_total_fpt"]].gt(0).all().all()
    ),
    "all_relative_gaps_finite_and_positive": bool(
        np.isfinite(levels["relative_gap"]).all() and levels["relative_gap"].gt(0).all()
    ),
    "relative_gap_larger_regionally_for_all_purposes": bool(
        relative_contrasts["regional_minus_district_relative_gap"].gt(0).all()
    ),
}
checks["gate_19_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "report FPT levels and assess scale-normalized scope amplification",
    "estimand": "female-minus-male mean FPT gap divided by the two-group mean FPT",
    "inference_boundary": (
        "Relative gaps are descriptive point diagnostics; the validated simultaneous "
        "inference remains attached to the absolute gaps."
    ),
    "identity_tolerance": identity_tolerance,
    "relative_scope_contrasts": json.loads(
        relative_contrasts.to_json(orient="records")
    ),
    "gate": checks,
}
(OUT / "gate_19.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series(checks, name="result").to_csv(OUT / "gate_19_checks.csv")

lines = [
    "# Scale-normalized diagnostic",
    "",
    ("The absolute female-minus-male FPT gaps are accompanied by group-specific "
     "mean levels and by a descriptive normalization using the two-group mean FPT."),
    "",
]
for row in relative_contrasts.itertuples(index=False):
    label = purpose_labels[row.purpose_group]
    lines.append(
        f"- {label}: relative gap {100*row.district_relative_gap:.1f}% in Bogotá D.C. "
        f"internal and {100*row.regional_relative_gap:.1f}% in Bogotá–Region; "
        f"regional-minus-district difference {100*row.regional_minus_district_relative_gap:.1f} percentage points."
    )
lines += [
    "",
    ("Interpretation: positive normalized differences for all three purposes show that "
     "the regional amplification is not explained solely by a higher overall FPT scale. "
     "These normalized quantities are descriptive and should not be assigned the absolute-gap confidence intervals."),
]
(OUT / "reviewer_m2_results.md").write_text("\n".join(lines), encoding="utf-8")

display(pd.Series(checks, name="result"))
if not checks["gate_19_passed"]:
    raise RuntimeError("Gate 19 failed; inspect gate_19_checks.csv")

persistent = DRIVE_BASE / "results/gate19"
persistent.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent / artifact.name)

print("Gate 19: APPROVED")
print("Persistent copy:", persistent)
print((OUT / "reviewer_m2_results.md").read_text(encoding="utf-8"))


,result
gate12_dependency_valid,True
six_scope_purpose_rows_present,True
all_required_scope_purpose_pairs_unique,True
absolute_gap_identity_valid,True
all_levels_finite_and_positive,True
all_relative_gaps_finite_and_positive,True
relative_gap_larger_regionally_for_all_purposes,True
gate_19_passed,True


Gate 19: APPROVED
Persistent copy: /content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023/results/gate19
# Scale-normalized diagnostic

The absolute female-minus-male FPT gaps are accompanied by group-specific mean levels and by a descriptive normalization using the two-group mean FPT.

- Employment: relative gap 12.4% in Bogotá D.C. internal and 21.7% in Bogotá–Region; regional-minus-district difference 9.2 percentage points.
- Education: relative gap 10.0% in Bogotá D.C. internal and 17.5% in Bogotá–Region; regional-minus-district difference 7.5 percentage points.
- Health: relative gap 4.7% in Bogotá D.C. internal and 10.7% in Bogotá–Region; regional-minus-district difference 6.0 percentage points.

Interpretation: positive normalized differences for all three purposes show that the regional amplification is not explained solely by a higher overall FPT scale. These normalized quantities are descriptive and should not be assigned the absolute-gap confidence intervals.
